# 프로파일링 및 타이밍 코드

코드를 개발하고 데이터 처리 파이프라인을 생성하는 과정에서 다양한 구현 간에 절충이 가능한 경우가 많습니다.
알고리즘 개발 초기에 그러한 것에 대해 걱정하는 것은 비생산적일 수 있습니다. Donald Knuth는 다음과 같이 유명하게 말했습니다. "약 97%의 경우 작은 효율성은 잊어야 합니다. 성급한 최적화는 모든 악의 근원입니다."

그러나 코드가 작동하게 되면 효율성을 조금 더 자세히 살펴보는 것이 유용할 수 있습니다.
때로는 특정 명령이나 명령 집합의 실행 시간을 확인하는 것이 유용할 수 있습니다. 다른 경우에는 여러 줄의 프로세스를 검사하고 복잡한 일련의 작업에서 병목 현상이 발생하는 위치를 확인하는 것이 유용합니다.
I파이썬(Python)은 이러한 종류의 타이밍 및 코드 프로파일링을 위한 다양한 기능에 대한 액세스를 제공합니다.
여기서는 다음 I파이썬(Python) 마법 명령에 대해 설명합니다.

- `%time`: 단일 명령문의 실행 시간입니다.
- `%timeit`: 정확성을 높이기 위해 단일 명령문을 반복 실행하는 시간
- `%prun`: 프로파일러를 사용하여 코드 실행
- `%lprun`: 라인별 프로파일러로 코드 실행
- `%memit`: 단일 문의 메모리 사용량을 측정합니다.
- `%mprun`: 라인별 메모리 프로파일러로 코드 실행

마지막 4개 명령은 I파이썬(Python)과 함께 번들로 제공되지 않습니다. 이를 사용하려면 'line_profiler' 및 'memory_profiler' 확장을 가져와야 합니다. 이에 대해서는 다음 섹션에서 설명하겠습니다.

## 타이밍 코드 조각: %timeit 및 %time

[I파이썬(Python) Magic Commands](01.03-Magic-Commands.ipynb)의 매직 기능 소개에서 `%timeit` 라인 매직과 `%%timeit` 셀 매직을 보았습니다. 이는 코드 조각의 반복 실행 시간을 측정하는 데 사용될 수 있습니다.

In [1]:
%timeit sum(range(100))

1.53 µs ± 47.8 ns per loop (mean ± std. dev. of 7 runs, 1000000 loops each)


이 작업은 매우 빠르기 때문에 `%timeit`은 자동으로 많은 반복을 수행합니다.
느린 명령의 경우 `%timeit`은 자동으로 더 적은 반복을 조정하고 수행합니다.

In [2]:
%%timeit
total = 0
for i in range(1000):
    for j in range(1000):
        total += i * (-1) ** j

536 ms ± 15.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


때로는 작업을 반복하는 것이 최선의 선택이 아닐 수도 있습니다.
예를 들어 정렬하려는 목록이 있는 경우 반복되는 작업으로 인해 오해를 받을 수 있습니다. 미리 정렬된 목록을 정렬하는 것은 정렬되지 않은 목록을 정렬하는 것보다 훨씬 빠르므로 반복하면 결과가 왜곡됩니다.

In [3]:
import random
L = [random.random() for i in range(100000)]
%timeit L.sort()

1.71 ms ± 334 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


이를 위해서는 `%time` 매직 함수가 더 나은 선택일 수 있습니다. 또한 짧은 시스템 관련 지연이 결과에 영향을 미칠 가능성이 거의 없는 장기 실행 명령에 대한 좋은 선택입니다.
정렬되지 않은 목록과 미리 정렬된 목록의 정렬 시간을 측정해 보겠습니다.

In [4]:
import random
L = [random.random() for i in range(100000)]
print("sorting an unsorted list:")
%time L.sort()

sorting an unsorted list:
CPU times: user 31.3 ms, sys: 686 µs, total: 32 ms
Wall time: 33.3 ms


In [5]:
print("sorting an already sorted list:")
%time L.sort()

sorting an already sorted list:
CPU times: user 5.19 ms, sys: 268 µs, total: 5.46 ms
Wall time: 14.1 ms


사전 정렬된 목록의 정렬 속도가 얼마나 빠른지 확인하세요. 또한 사전 정렬된 목록의 경우에도 `%time`과 `%timeit`의 타이밍이 얼마나 오래 걸리는지 확인하세요!
이는 '%timeit'이 시스템 호출이 타이밍을 방해하는 것을 방지하기 위해 내부적으로 몇 가지 영리한 작업을 수행한다는 사실의 결과입니다.
예를 들어 타이밍에 영향을 줄 수 있는 사용되지 않는 파이썬(Python) 객체(*가비지 수집*이라고도 함)를 정리하는 것을 방지합니다.
이러한 이유로 `%timeit` 결과는 일반적으로 `%time` 결과보다 눈에 띄게 빠릅니다.

`%time`의 경우 `%timeit`과 마찬가지로 `%%` 셀 매직 구문을 사용하면 여러 줄 스크립트의 타이밍을 지정할 수 있습니다.

In [6]:
%%time
total = 0
for i in range(1000):
    for j in range(1000):
        total += i * (-1) ** j

CPU times: user 655 ms, sys: 5.68 ms, total: 661 ms
Wall time: 710 ms


'%time' 및 '%timeit'과 사용 가능한 옵션에 대한 자세한 내용을 보려면 I파이썬(Python) 도움말 기능을 사용하세요(예: I파이썬(Python) 프롬프트에서 '%time?' 입력).

## 전체 스크립트 프로파일링: %prun

프로그램은 여러 개의 단일 명령문으로 구성되며 때로는 문맥에 따라 이러한 명령문의 타이밍을 맞추는 것이 자체적으로 타이밍을 맞추는 것보다 더 중요합니다.
파이썬(Python)에는 내장 코드 프로파일러(파이썬(Python) 문서에서 읽을 수 있음)가 포함되어 있지만 I파이썬(Python)은 마법 함수 `%prun` 형식으로 이 프로파일러를 사용하는 훨씬 더 편리한 방법을 제공합니다.

예를 들어 몇 가지 계산을 수행하는 간단한 함수를 정의하겠습니다.

In [7]:
def sum_of_lists(N):
    total = 0
    for i in range(5):
        L = [j ^ (j >> i) for j in range(N)]
        total += sum(L)
    return total

이제 프로파일링된 결과를 보기 위해 함수 호출로 `%prun`을 호출할 수 있습니다.

In [8]:
%prun sum_of_lists(1000000)

         14 function calls in 0.932 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        5    0.808    0.162    0.808    0.162 <ipython-input-7-f105717832a2>:4(<listcomp>)
        5    0.066    0.013    0.066    0.013 {built-in method builtins.sum}
        1    0.044    0.044    0.918    0.918 <ipython-input-7-f105717832a2>:1(sum_of_lists)
        1    0.014    0.014    0.932    0.932 <string>:1(<module>)
        1    0.000    0.000    0.932    0.932 {built-in method builtins.exec}
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}

결과는 각 함수 호출의 총 시간 순으로 실행에 가장 많은 시간이 소요되는 부분을 나타내는 테이블입니다. 이 경우 실행 시간의 대부분은 `sum_of_lists` 내부의 목록 이해에 있습니다.
여기에서 우리는 알고리즘의 성능을 향상시키기 위해 어떤 변경을 할 수 있는지 생각해 살펴볼 수 있습니다.

'%prun' 및 사용 가능한 옵션에 대한 자세한 내용을 보려면 I파이썬(Python) 도움말 기능을 사용하세요(예: I파이썬(Python) 프롬프트에서 '%prun?' 입력).

## %lprun을 사용한 라인별 프로파일링

`%prun`의 기능별 프로파일링은 유용하지만 때로는 행별 프로파일 보고서를 갖는 것이 더 편리합니다.
이는 파이썬(Python)이나 I파이썬(Python)에 내장되어 있지 않지만 이를 수행할 수 있는 설치용 `line_profiler` 패키지가 있습니다.
먼저 파이썬(Python)의 패키징 도구인 `pip`를 사용하여 `line_profiler` 패키지를 설치하세요.

````
$ pip 설치 line_profiler
````

다음으로, I파이썬(Python)을 사용하여 이 패키지의 일부로 제공되는 `line_profiler` I파이썬(Python) 확장을 로드할 수 있습니다.

In [9]:
%load_ext line_profiler

이제 `%lprun` 명령은 모든 함수에 대해 한 줄씩 프로파일링을 수행합니다. 이 경우 프로파일링에 관심이 있는 함수를 명시적으로 알려야 합니다.

In [10]:
%lprun -f sum_of_lists sum_of_lists(5000)

Timer unit: 1e-06 s

Total time: 0.014803 s
File: <ipython-input-7-f105717832a2>
Function: sum_of_lists at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def sum_of_lists(N):
     2         1          6.0      6.0      0.0      total = 0
     3         6         13.0      2.2      0.1      for i in range(5):
     4         5      14242.0   2848.4     96.2          L = [j ^ (j >> i) for j in range(N)]
     5         5        541.0    108.2      3.7          total += sum(L)
     6         1          1.0      1.0      0.0      return total

상단의 정보는 결과를 읽는 데 중요한 정보를 제공합니다. 시간은 마이크로초 단위로 보고되며 프로그램이 가장 많은 시간을 소비하는 위치를 확인할 수 있습니다.
이 시점에서 우리는 이 정보를 사용하여 스크립트의 측면을 수정하고 원하는 사용 사례에 맞게 더 나은 성능을 발휘하도록 만들 수 있습니다.

'%lprun' 및 사용 가능한 옵션에 대한 자세한 내용을 보려면 I파이썬(Python) 도움말 기능을 사용하세요(예: I파이썬(Python) 프롬프트에서 '%lprun?' 입력).

## 메모리 사용 프로파일링: %memit 및 %mprun

프로파일링의 또 다른 측면은 작업에 사용되는 메모리 양입니다.
이는 또 다른 I파이썬(Python) 확장인 `memory_profiler`를 사용하여 평가할 수 있습니다.
`line_profiler`와 마찬가지로 `pip` 확장 프로그램 설치부터 시작합니다.

````
$ pip 설치 memory_profiler
````

그런 다음 I파이썬(Python)을 사용하여 로드할 수 있습니다.

In [11]:
%load_ext memory_profiler

메모리 프로파일러 확장에는 두 가지 유용한 마법 함수가 포함되어 있습니다. `%memit`(`%timeit`에 해당하는 메모리 측정 제공) 및 `%mprun`(`%lprun`에 해당하는 메모리 측정 제공).
`%memit` 마법 함수는 오히려 간단하게 사용할 수 있습니다:

In [12]:
%memit sum_of_lists(1000000)

peak memory: 141.70 MiB, increment: 75.65 MiB


이 함수는 약 140MB의 메모리를 사용하는 것을 살펴볼 수 있습니다.

메모리 사용을 한 줄씩 설명하려면 `%mprun` 마법 함수를 사용할 수 있습니다.
불행하게도 이것은 노트북 자체가 아닌 별도의 모듈에 정의된 함수에 대해서만 작동하므로 먼저 `%%file` 셀 매직을 사용하여 `sum_of_lists` 함수가 포함된 `mprun_demo.py`라는 간단한 모듈을 생성하고 메모리 프로파일링 결과를 더 명확하게 만드는 한 가지 추가 사항을 추가하겠습니다.

In [13]:
%%file mprun_demo.py
def sum_of_lists(N):
    total = 0
    for i in range(5):
        L = [j ^ (j >> i) for j in range(N)]
        total += sum(L)
        del L # remove reference to L
    return total

Overwriting mprun_demo.py


이제 이 함수의 새 버전을 가져오고 메모리 라인 프로파일러를 실행할 수 있습니다.

In [14]:
from mprun_demo import sum_of_lists
%mprun -f sum_of_lists sum_of_lists(1000000)

Filename: /Users/jakevdp/github/jakevdp/PythonDataScienceHandbook/notebooks_v2/mprun_demo.py

Line #    Mem usage    Increment  Occurences   Line Contents
     1     66.7 MiB     66.7 MiB           1   def sum_of_lists(N):
     2     66.7 MiB      0.0 MiB           1       total = 0
     3     75.1 MiB      8.4 MiB           6       for i in range(5):
     4    105.9 MiB     30.8 MiB     5000015           L = [j ^ (j >> i) for j in range(N)]
     5    109.8 MiB      3.8 MiB           5           total += sum(L)
     6     75.1 MiB    -34.6 MiB           5           del L # remove reference to L
     7     66.9 MiB     -8.2 MiB           1       return total

여기서 `Increment` 열은 각 행이 전체 메모리 예산에 얼마나 영향을 미치는지 알려줍니다. `L` 목록을 생성하고 삭제할 때 약 30MB의 메모리 사용량이 추가된다는 점을 관찰하세요.
이는 파이썬(Python) 인터프리터 자체의 백그라운드 메모리 사용량에 더해집니다.

'%memit' 및 '%mprun'과 사용 가능한 옵션에 대한 자세한 내용을 보려면 I파이썬(Python) 도움말 기능을 사용하세요(예: I파이썬(Python) 프롬프트에서 '%memit?' 입력).